# Jev basics

The three Jev question types — `Choice`, `Score`, `Noul` — in one request. Jev classifies; plain Python decides.

Needs `OPENROUTER_API_KEY` in the environment. Install: `pip install typesafe-sdk`.

In [ ]:
# !pip install typesafe_sdk

In [ ]:
import os
from typesafe_sdk import TypeSafeClient, Choice, Score, Noul

jev = TypeSafeClient(api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api")
MODEL = "~typesafe/jev-latest"

## Choice — pick one option

In [ ]:
ans = jev.system_one(
    model=MODEL,
    state="I was charged twice and need a refund.",
    questions={
        "department": Choice(
            instructions="Which team should handle this?",
            criteria={  
                "Billing": None, 
                "Technical Support": None, 
                "Sales": None, 
                "Other": None
            },
        )
    },
).answers["department"]

print(ans.choice, ans.confidence)

## Score — rate on an ordered rubric

In [ ]:
ans = jev.system_one(
    model=MODEL,
    state="This is the THIRD time I've been double-charged. Fix it now.",
    questions={
        "frustration": Score(
            instructions="How frustrated is the customer?",
            criteria=["Calm", "Mildly annoyed", "Very angry"],
        )
    },
).answers["frustration"]

print(ans.score, ans.legend[round(ans.score)])

In [ ]:
ans

## Noul — one yes/no gate

`.noul` is the probability of "yes".

In [ ]:
ans = jev.system_one(
    model=MODEL,
    state="Refund me now or I'm disputing every charge with my bank.",
    questions={
        "needs_human": Noul(
            instructions="Does this need a human agent?",
            # criteria={"true": "Angry or high-stakes.", "false": "Routine and safe to automate."},
        )
    },
).answers["needs_human"]

print(ans.noul)

## All three at once, then decide in plain Python

In [ ]:
a = jev.system_one(
    model=MODEL,
    state="This is the THIRD time I've been double-charged. Refund me now or I'm disputing with my bank.",
    questions={
        "department": Choice(
            instructions="Which team should handle this?",
            criteria={"Billing": None, "Technical Support": None, "Sales": None, "Other": None},
        ),
        "frustration": Score(
            instructions="How frustrated is the customer?",
            criteria=["Calm", "Mildly annoyed", "Very angry"],
        ),
        "needs_human": Noul(instructions="Does this need a human agent?"),
    },
).answers

escalate = a["needs_human"].noul >= 0.6 or a["frustration"].score >= 1.5
if escalate:
    print(f"Escalate to a human on the {a['department'].choice} team.")
else:
    print(f"Auto-reply, routed to {a['department'].choice}.")